## DW Assignment 4: Paper Review and Algorithm Evaluation—Algorithmic Fairness
### Author: Daša Jalen

This Python notebook includes the code for DW Assignment 4 Task 2: Evaluating the effectiveness of the algorithm. The selected bias mitigation method is reweighting, a pre-processing algorithm available in the AIF360 package (Bellamy et al., 2019) and presented in the paper *Building Classifiers with Independency Constraints* (Calders et al., 2009). The algorithm will be applied before training a simple logistic regression classifier on three separate datasets.

In [1]:
# loading the packages
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from sklearn.model_selection import train_test_split
from aif360.algorithms.preprocessing import Reweighing
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


In [2]:
# loading the data
adult = pd.read_csv("data/adult.csv")
compas = pd.read_csv("data/propublica_data_for_fairml.csv")
german = fetch_ucirepo(id=144).data.original

### Data exploration

In [3]:
adult.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
compas.head()

,Two_yr_Recidivism,Number_of_Priors,score_factor,Age_Above_FourtyFive,Age_Below_TwentyFive,African_American,Asian,Hispanic,Native_American,Other,Female,Misdemeanor
0,0,0,0,1,0,0,0,0,0,1,0,0
1,1,0,0,0,0,1,0,0,0,0,0,0
2,1,4,0,0,1,1,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,0,1
4,1,14,1,0,0,0,0,0,0,0,0,0


In [5]:
german.head()

,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [6]:
# inspecting the dimensions and variables
print(adult.columns)
print(adult.shape)
print(compas.columns)
print(compas.shape)
print(german.columns)
print(german.shape)

Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'income'],
      dtype='str')
(32561, 15)
Index(['Two_yr_Recidivism', 'Number_of_Priors', 'score_factor',
       'Age_Above_FourtyFive', 'Age_Below_TwentyFive', 'African_American',
       'Asian', 'Hispanic', 'Native_American', 'Other', 'Female',
       'Misdemeanor'],
      dtype='str')
(6172, 12)
Index(['Attribute1', 'Attribute2', 'Attribute3', 'Attribute4', 'Attribute5',
       'Attribute6', 'Attribute7', 'Attribute8', 'Attribute9', 'Attribute10',
       'Attribute11', 'Attribute12', 'Attribute13', 'Attribute14',
       'Attribute15', 'Attribute16', 'Attribute17', 'Attribute18',
       'Attribute19', 'Attribute20', 'class'],
      dtype='str')
(1000, 21)


In [7]:
# we remove fnlwgt (final weight) from adult, which is not relevant for the task
adult = adult.drop("fnlwgt", axis = 1)

In [8]:
# the feature names in the german dataframe are uninformative; the following function returns more information
fetch_ucirepo(id=144).variables

,name,role,type,demographic,description,units,missing_values
0,Attribute1,Feature,Categorical,NaN,Status of existing checking account,NaN,no
1,Attribute2,Feature,Integer,NaN,Duration,months,no
2,Attribute3,Feature,Categorical,NaN,Credit history,NaN,no
3,Attribute4,Feature,Categorical,NaN,Purpose,NaN,no
4,Attribute5,Feature,Integer,NaN,Credit amount,NaN,no
5,Attribute6,Feature,Categorical,NaN,Savings account/bonds,NaN,no
6,Attribute7,Feature,Categorical,Other,Present employment since,NaN,no
7,Attribute8,Feature,Integer,NaN,Installment rate in percentage of disposable i...,NaN,no
8,Attribute9,Feature,Categorical,Marital Status,Personal status and sex,NaN,no
9,Attribute10,Feature,Categorical,NaN,Other debtors / guarantors,NaN,no


In [9]:
print(adult.describe(include = "all").round(2))
print(compas.describe(include = "all").round(2))
print(german.describe(include = "all").round(2))

             age workclass education  education.num      marital.status  \
count   32561.00     32561     32561       32561.00               32561   
unique       NaN         9        16            NaN                   7   
top          NaN   Private   HS-grad            NaN  Married-civ-spouse   
freq         NaN     22696     10501            NaN               14976   
mean       38.58       NaN       NaN          10.08                 NaN   
std        13.64       NaN       NaN           2.57                 NaN   
min        17.00       NaN       NaN           1.00                 NaN   
25%        28.00       NaN       NaN           9.00                 NaN   
50%        37.00       NaN       NaN          10.00                 NaN   
75%        48.00       NaN       NaN          12.00                 NaN   
max        90.00       NaN       NaN          16.00                 NaN   

            occupation relationship   race    sex  capital.gain  capital.loss  \
count            3

In [10]:
# checking for missing values
print(adult.isnull().sum())
print(compas.isnull().sum())
print(german.isnull().sum())

age               0
workclass         0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64
Two_yr_Recidivism       0
Number_of_Priors        0
score_factor            0
Age_Above_FourtyFive    0
Age_Below_TwentyFive    0
African_American        0
Asian                   0
Hispanic                0
Native_American         0
Other                   0
Female                  0
Misdemeanor             0
dtype: int64
Attribute1     0
Attribute2     0
Attribute3     0
Attribute4     0
Attribute5     0
Attribute6     0
Attribute7     0
Attribute8     0
Attribute9     0
Attribute10    0
Attribute11    0
Attribute12    0
Attribute13    0
Attribute14    0
Attribute15    0
Attribute16    0
Attribute17    0
Attribute18    0
Attribute19    0
Attribute20    0
class          0
dtype: int64


All three datasets are very clean and contain no unusual or missing values.

### Pre-processing
We start by transforming the target variables and the protected attributes into binary variables, with 1 - favorable outcome, 0 - unfavorable outcome and 1 - privileged group, 0 - unprivileged group. The following features will be used:

**Target variables:**  
adult: *income* (<=50K to 0; >50K to 1)  
compas: *Two_yr_Recidivism* (1/recidivism to 0; 0/no recidivism to 1)  
german: *class* - good or bad credit risks (1/good to 1; 2/bad to 0)

**Protected attributes:**  
adult: *sex* (Female to 0; Male to 1)  
compas: *African_American* (1/African_American to 0; 0/Other to 1)  
german: *Attribute9* - personal status and sex (A91, A93, A94/male to 1; A92, A95/female to 0)  

In [11]:
adult["income_binary"] = adult["income"].map({
    "<=50K": 0,
    ">50K": 1
})

adult["sex_binary"] = adult["sex"].map({
    "Female": 0,
    "Male": 1
})

In [12]:
adult[["income_binary", "sex_binary"]].isna().sum() # checking that no missing values were created

income_binary    0
sex_binary       0
dtype: int64

In [13]:
adult = adult.drop(["income", "sex"], axis = 1) # removing original columns

In [14]:
compas["rec_binary"] = compas["Two_yr_Recidivism"].map({
    1: 0,
    0: 1
})

compas["African_American_binary"] = compas["African_American"].map({
    1: 0,
    0: 1
})

In [15]:
compas[["rec_binary", "African_American_binary"]].isna().sum()

rec_binary                 0
African_American_binary    0
dtype: int64

In [16]:
compas = compas.drop(["African_American", "Two_yr_Recidivism"], axis = 1)

In [17]:
german["class_binary"] = german["class"].map({
    1: 1,
    2: 0
})

german["sex_binary"] = german["Attribute9"].map({
    "A91": 1,
    "A93": 1,
    "A94": 1,
    "A92": 0,
    "A95": 0
})

In [18]:
german[["class_binary", "sex_binary"]].isna().sum()

class_binary    0
sex_binary      0
dtype: int64

In [19]:
german = german.drop(["Attribute9", "class"], axis = 1)

Now we transform all non-numerical attributes into numerical, which are compatible with the logistic regression model.

In [20]:
adult.dtypes

age               int64
workclass           str
education           str
education.num     int64
marital.status      str
occupation          str
relationship        str
race                str
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country      str
income_binary     int64
sex_binary        int64
dtype: object

In [21]:
adult = adult.drop("education", axis = 1) # we drop education, because we already have education.num

adult_numeric = pd.get_dummies(
    adult,
    columns = ["workclass", "marital.status", "occupation", "relationship", "race", "native.country"],
    dtype = int
)

In [22]:
compas.dtypes # already integers

Number_of_Priors           int64
score_factor               int64
Age_Above_FourtyFive       int64
Age_Below_TwentyFive       int64
Asian                      int64
Hispanic                   int64
Native_American            int64
Other                      int64
Female                     int64
Misdemeanor                int64
rec_binary                 int64
African_American_binary    int64
dtype: object

In [23]:
compas_numeric = compas

In [24]:
german.dtypes

Attribute1        str
Attribute2      int64
Attribute3        str
Attribute4        str
Attribute5      int64
Attribute6        str
Attribute7        str
Attribute8      int64
Attribute10       str
Attribute11     int64
Attribute12       str
Attribute13     int64
Attribute14       str
Attribute15       str
Attribute16     int64
Attribute17       str
Attribute18     int64
Attribute19       str
Attribute20       str
class_binary    int64
sex_binary      int64
dtype: object

In [25]:
german_numeric = pd.get_dummies(
    german,
    columns = ["Attribute1", "Attribute3", "Attribute4", "Attribute6", "Attribute7", "Attribute10", "Attribute12", "Attribute14", "Attribute15",
               "Attribute17", "Attribute19", "Attribute20"],
    dtype = int
)

Before computing disparate impact ratio (DIR), demographic parity difference (DP) and consistency, we scale numerical and ordinal features so that the scale of the attributes does not impact the results. This is important because consistency measure is based on distances (it uses k-NN).

In [26]:
adult_scaled = adult_numeric.copy()

features = ["age", "education.num", "capital.gain", "capital.loss", "hours.per.week"]

scaler = StandardScaler()

adult_scaled[features] = scaler.fit_transform(adult_scaled[features])

In [27]:
compas_scaled = compas_numeric.copy()

features = ["Number_of_Priors"]

scaler = StandardScaler()

compas_scaled[features] = scaler.fit_transform(compas_scaled[features])

In [28]:
german_scaled = german_numeric.copy()

features = ["Attribute2", "Attribute5", "Attribute8", "Attribute11", "Attribute13", "Attribute16", "Attribute18"]

scaler = StandardScaler()

german_scaled[features] = scaler.fit_transform(german_scaled[features])

### Fairness measures before mitigation
The datasets have been prepared so we can now compute the demographic parity difference (DP_Diff), the disparate impact ratio (DIR), and the consistency from the original labels before training the logistic regression classifier. We will use the AIF360 package, so we first have to transform the Pandas dataframes into datasets compatible with this library.

In [29]:
adult_dataset = BinaryLabelDataset(
    df = adult_scaled,
    label_names = ["income_binary"],
    protected_attribute_names = ["sex_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

In [30]:
compas_dataset = BinaryLabelDataset(
    df = compas_scaled,
    label_names = ["rec_binary"],
    protected_attribute_names = ["African_American_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

In [31]:
german_dataset = BinaryLabelDataset(
    df = german_scaled,
    label_names = ["class_binary"],
    protected_attribute_names = ["sex_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

Now we can compute the three fairness measures.

In [32]:
adult_metrics = BinaryLabelDatasetMetric(
    adult_dataset,
    unprivileged_groups = [{"sex_binary": 0}],
    privileged_groups = [{"sex_binary": 1}]
)

print("DP:", round(adult_metrics.statistical_parity_difference(), 2))
print("DIR:", round(adult_metrics.disparate_impact(), 2))
print("consistency:", round(adult_metrics.consistency().item(), 2))

DP: -0.2
DIR: 0.36
consistency: 0.84


In [33]:
compas_metrics = BinaryLabelDatasetMetric(
    compas_dataset,
    unprivileged_groups = [{"African_American_binary": 0}],
    privileged_groups = [{"African_American_binary": 1}]
)

print("DP:", round(compas_metrics.statistical_parity_difference(), 2))
print("DIR:", round(compas_metrics.disparate_impact(), 2))
print("Consistency:", round(compas_metrics.consistency().item(), 2))

DP: -0.14
DIR: 0.77
Consistency: 0.63


In [34]:
german_metrics = BinaryLabelDatasetMetric(
    german_dataset,
    unprivileged_groups = [{"sex_binary": 0}],
    privileged_groups = [{"sex_binary": 1}]
)

print("DP:", round(german_metrics.statistical_parity_difference(), 2))
print("DIR:", round(german_metrics.disparate_impact(), 2))
print("Consistency:", round(german_metrics.consistency().item(), 2))

DP: -0.07
DIR: 0.9
Consistency: 0.74


### Logistic regression with reweighting
Now we will apply the logistic regression classifier with weights to our three datasets. We will use accuracy as a performance measure and compute the three fairness metrics again using the model predicted values. The workflow will be as follows:
1. Perform the train/test split with stratification.
2. Apply the scaler on the training data and use its learned parameters on the test data to avoid data leakage.
3. Obtain the weights from the AIF360 package.
4. Train the logistic classifier with those weights.
5. Predict the outcome for the test data.
6. Compute the accuracy, demographic parity difference, disparate impact ratio (DIR) and the consistency for the final predictions.

We will go through this entire pipeline for each dataset separately.

#### Adult

In [35]:
# train/test split

adult_x = adult_numeric.drop("income_binary", axis = 1)
adult_y = adult_numeric["income_binary"]

adult_x_train, adult_x_test, adult_y_train, adult_y_test = train_test_split(
    adult_x, adult_y, test_size = 0.2, stratify = adult_y, random_state = 123)

# scaling

adult_x_train = adult_x_train.copy()
adult_x_test = adult_x_test.copy()

features = ["age", "education.num", "capital.gain", "capital.loss", "hours.per.week"]

scaler = StandardScaler()

adult_x_train[features] = scaler.fit_transform(adult_x_train[features])
adult_x_test[features] = scaler.transform(adult_x_test[features])

Now we create a training dataset that is compatible with the AIF360 library.

In [36]:
adult_training_df = pd.concat([adult_x_train, adult_y_train], axis = 1)

adult_training_dataset = BinaryLabelDataset(
    df = adult_training_df,
    label_names = ["income_binary"],
    protected_attribute_names = ["sex_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

We apply the reweighting.

In [37]:
adult_reweighter = Reweighing(
    unprivileged_groups = [{"sex_binary": 0}],
    privileged_groups = [{"sex_binary": 1}]
)

adult_training_reweighted = adult_reweighter.fit_transform(adult_training_dataset)

adult_weights = adult_training_reweighted.instance_weights

Now we can train the logistic classifier with the learned weights.

In [38]:
adult_model = LogisticRegression(max_iter = 1000)

adult_model.fit(
    adult_x_train,
    adult_y_train,
    sample_weight = adult_weights
)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

Let's predict the test data outcome (binary income).

In [39]:
adult_predictions = adult_model.predict(adult_x_test)

We compute the accuracy score.

In [40]:
adult_accuracy = accuracy_score(adult_y_test, adult_predictions)
print("Accuracy:", round(adult_accuracy, 2))

Accuracy: 0.85


Finally, we can compute the three fairness metrics for the predicted values after applying our bias mitigation algorithm. Note that we have to create an AIF360 compatible dataset again.

In [41]:
adult_test_df = adult_x_test.copy()

adult_test_df["income_binary"] = adult_predictions

adult_test_dataset = BinaryLabelDataset(
    df = adult_test_df,
    label_names = ["income_binary"],
    protected_attribute_names = ["sex_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

In [42]:
adult_test_metrics = BinaryLabelDatasetMetric(
    adult_test_dataset,
    unprivileged_groups = [{"sex_binary": 0}],
    privileged_groups = [{"sex_binary": 1}]
)

print("DP", round(adult_test_metrics.statistical_parity_difference(), 2))
print("DIR:", round(adult_test_metrics.disparate_impact(), 2))
print("Consistency:", round(adult_test_metrics.consistency().item(), 2))

DP -0.09
DIR: 0.6
Consistency: 0.96


#### Compas

In [43]:
# train/test split

compas_x = compas_numeric.drop("rec_binary", axis = 1)
compas_y = compas_numeric["rec_binary"]

compas_x_train, compas_x_test, compas_y_train, compas_y_test = train_test_split(
    compas_x, compas_y, test_size = 0.2, stratify = compas_y, random_state = 123)

# scaling

compas_x_train = compas_x_train.copy()
compas_x_test = compas_x_test.copy()

features = ["Number_of_Priors"]

scaler = StandardScaler()

compas_x_train[features] = scaler.fit_transform(compas_x_train[features])
compas_x_test[features] = scaler.transform(compas_x_test[features])

In [44]:
compas_training_df = pd.concat([compas_x_train, compas_y_train], axis = 1)

compas_training_dataset = BinaryLabelDataset(
    df = compas_training_df,
    label_names = ["rec_binary"],
    protected_attribute_names = ["African_American_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

In [45]:
compas_reweighter = Reweighing(
    unprivileged_groups = [{"African_American_binary": 0}],
    privileged_groups = [{"African_American_binary": 1}]
)

compas_training_reweighted = compas_reweighter.fit_transform(compas_training_dataset)

compas_weights = compas_training_reweighted.instance_weights

In [46]:
compas_model = LogisticRegression(max_iter = 1000)

compas_model.fit(
    compas_x_train,
    compas_y_train,
    sample_weight = compas_weights
)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [47]:
compas_predictions = compas_model.predict(compas_x_test)

In [48]:
compas_accuracy = accuracy_score(compas_y_test, compas_predictions)
print("Accuracy:", round(compas_accuracy, 2))

Accuracy: 0.67


In [49]:
compas_test_df = compas_x_test.copy()

compas_test_df["rec_binary"] = compas_predictions

compas_test_dataset = BinaryLabelDataset(
    df = compas_test_df,
    label_names = ["rec_binary"],
    protected_attribute_names = ["African_American_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

In [50]:
compas_test_metrics = BinaryLabelDatasetMetric(
    compas_test_dataset,
    unprivileged_groups = [{"African_American_binary": 0}],
    privileged_groups = [{"African_American_binary": 1}]
)

print("DP:", round(compas_test_metrics.statistical_parity_difference(), 2))
print("DIR:", round(compas_test_metrics.disparate_impact(), 2))
print("Consistency:", round(compas_test_metrics.consistency().item(), 2))

DP: -0.13
DIR: 0.82
Consistency: 0.98


#### German

In [51]:
# train/test split

german_x = german_numeric.drop("class_binary", axis = 1)
german_y = german_numeric["class_binary"]

german_x_train, german_x_test, german_y_train, german_y_test = train_test_split(
    german_x, german_y, test_size = 0.2, stratify = german_y, random_state = 123)

# scaling

german_x_train = german_x_train.copy()
german_x_test = german_x_test.copy()

features = ["Attribute2", "Attribute5", "Attribute8", "Attribute11", "Attribute13", "Attribute16", "Attribute18"]

scaler = StandardScaler()

german_x_train[features] = scaler.fit_transform(german_x_train[features])
german_x_test[features] = scaler.transform(german_x_test[features])

In [52]:
german_training_df = pd.concat([german_x_train, german_y_train], axis = 1)

german_training_dataset = BinaryLabelDataset(
    df = german_training_df,
    label_names = ["class_binary"],
    protected_attribute_names = ["sex_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

In [53]:
german_reweighter = Reweighing(
    unprivileged_groups = [{"sex_binary": 0}],
    privileged_groups = [{"sex_binary": 1}]
)

german_training_reweighted = german_reweighter.fit_transform(german_training_dataset)

german_weights = german_training_reweighted.instance_weights

In [54]:
german_model = LogisticRegression(max_iter = 1000)

german_model.fit(
    german_x_train,
    german_y_train,
    sample_weight = german_weights
)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [55]:
german_predictions = german_model.predict(german_x_test)

In [56]:
german_accuracy = accuracy_score(german_y_test, german_predictions)
print("Accuracy:", round(german_accuracy, 2))

Accuracy: 0.73


In [57]:
german_test_df = german_x_test.copy()

german_test_df["class_binary"] = german_predictions

german_test_dataset = BinaryLabelDataset(
    df = german_test_df,
    label_names = ["class_binary"],
    protected_attribute_names = ["sex_binary"],
    favorable_label = 1,
    unfavorable_label = 0
)

In [58]:
german_test_metrics = BinaryLabelDatasetMetric(
    german_test_dataset,
    unprivileged_groups = [{"sex_binary": 0}],
    privileged_groups = [{"sex_binary": 1}]
)

print("DP", round(german_test_metrics.statistical_parity_difference(), 2))
print("DIR:", round(german_test_metrics.disparate_impact(), 2))
print("Consistency:", round(german_test_metrics.consistency().item(), 2))

DP -0.05
DIR: 0.94
Consistency: 0.82


All fairness metrics improved for all three datasets: demographic parity difference towards 0, disparate impact ratio and consistency towards 1. The reweighting procedure has been successful.

### References

Bellamy, R. K. E., Dey, K., Hind, M., Hoffman, S. C., Houde, S., Kannan, K., Lohia, P., Martino, J., Mehta, S., Mojsilović, S., Nagar, S., Ramamurthy, K. N., Richards, J., Saha, D., Sattigeri, P., Singh, M., Varshney, K. R., & Zhang, Y. (2019). AI Fairness 360: An extensible toolkit for detecting and mitigating algorithmic bias. *IBM Journal of Research and Development, 63*(4/5), 4:1–4:15. https://doi.org/10.1147/JRD.2019.2942287

Calders, T., Kamiran, F., & Pechenizkiy, M. (2009). Building classifiers with independency constraints. In *2009 IEEE International Conference on Data Mining Workshops* (pp. 13–18). https://doi.org/10.1109/ICDMW.2009.83
